In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tqdm import tqdm

In [ ]:
TRAIN_DIR = "./brain_tumor/Dataset/classification_data/Processed/train"
VAL_DIR   = "./brain_tumor/Dataset/classification_data/Processed/val"
TEST_DIR  = "./brain_tumor/Dataset/classification_data/Processed/test"

IMAGE_SIZE = 224

BATCH_SIZE = 16

LEARNING_RATE = 1e-4

EPOCHS = 20

CHECKPOINT_DIR = "checkpoints"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

device = torch.device("cpu")

print(device)

cpu


In [3]:
class BrainTumourDataset(Dataset):

    def __init__(
        self,
        root_dir,
        image_size=224
    ):

        self.root_dir = root_dir
        self.image_size = image_size

        self.classes = sorted(
            [
                folder
                for folder in os.listdir(root_dir)
                if os.path.isdir(
                    os.path.join(root_dir, folder)
                )
            ]
        )

        self.class_to_idx = {
            cls: idx
            for idx, cls in enumerate(
                self.classes
            )
        }

        self.samples = []

        for cls_name in self.classes:

            cls_dir = os.path.join(
                root_dir,
                cls_name
            )

            for img_name in os.listdir(
                cls_dir
            ):

                if img_name.lower().endswith(
                    (
                        ".png",
                        ".jpg",
                        ".jpeg"
                    )
                ):

                    self.samples.append(
                        (
                            os.path.join(
                                cls_dir,
                                img_name
                            ),
                            self.class_to_idx[
                                cls_name
                            ]
                        )
                    )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = cv2.imread(
            img_path,
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:

            return self.__getitem__(
                (idx + 1)
                % len(self.samples)
            )

        image = cv2.resize(
            image,
            (
                self.image_size,
                self.image_size
            )
        )

        image = (
            image.astype(np.float32)
            / 255.0
        )

        image = torch.tensor(
            image,
            dtype=torch.float32
        ).unsqueeze(0)

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return image, label

In [4]:
train_dataset = BrainTumourDataset(
    TRAIN_DIR,
    IMAGE_SIZE
)

val_dataset = BrainTumourDataset(
    VAL_DIR,
    IMAGE_SIZE
)

test_dataset = BrainTumourDataset(
    TEST_DIR,
    IMAGE_SIZE
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(
    "Train Samples:",
    len(train_dataset)
)

print(
    "Val Samples:",
    len(val_dataset)
)

print(
    "Test Samples:",
    len(test_dataset)
)

print(
    train_dataset.class_to_idx
)

Train Samples: 6442
Val Samples: 1613
Test Samples: 2208
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [ ]:
class BrainTumourCNN(nn.Module):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(
                64,
                128,
                3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(
                128,
                256,
                3,
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.AdaptiveAvgPool2d(
                (1, 1)
            )
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [ ]:
model = BrainTumourCNN(
    num_classes=len(
        train_dataset.classes
    )
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

In [7]:
def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    best_val_loss,
    filename
):

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "scheduler_state_dict":
                scheduler.state_dict(),
            "best_val_loss":
                best_val_loss
        },
        filename
    )


def load_checkpoint(
    filename
):

    checkpoint = torch.load(
        filename,
        map_location=device
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )

    scheduler.load_state_dict(
        checkpoint[
            "scheduler_state_dict"
        ]
    )

    return (
        checkpoint["epoch"],
        checkpoint["best_val_loss"]
    )

In [8]:
history = {
    "train_loss": [],
    "val_loss": [],
    "val_acc": []
}

best_val_loss = float("inf")

early_stop_counter = 0

early_stop_patience = 4

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for images, labels in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    ):

        images = images.to(device)

        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss = (
        train_loss
        / len(train_loader)
    )

    model.eval()

    val_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            preds = outputs.argmax(
                dim=1
            )

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    val_loss = (
        val_loss
        / len(val_loader)
    )

    val_acc = (
        correct
        / total
    )

    scheduler.step(val_loss)

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_acc"].append(
        val_acc
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Acc: {val_acc:.4f}"
    )

    save_checkpoint(
        epoch,
        model,
        optimizer,
        scheduler,
        best_val_loss,
        os.path.join(
            CHECKPOINT_DIR,
            "latest_checkpoint.pth"
        )
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        early_stop_counter = 0

        save_checkpoint(
            epoch,
            model,
            optimizer,
            scheduler,
            best_val_loss,
            os.path.join(
                CHECKPOINT_DIR,
                "best_model.pth"
            )
        )

        print(
            "Best model saved."
        )

    else:

        early_stop_counter += 1

        print(
            f"No improvement ({early_stop_counter}/{early_stop_patience})"
        )

    if early_stop_counter >= early_stop_patience:

        print(
            "Early stopping triggered."
        )

        break

Epoch 1/20: 100%|██████████| 403/403 [14:58<00:00,  2.23s/it]


Train Loss: 0.7834
Val Loss: 0.8129
Val Acc: 0.6528
Best model saved.


Epoch 2/20: 100%|██████████| 403/403 [12:18<00:00,  1.83s/it]


Train Loss: 0.6110
Val Loss: 0.6435
Val Acc: 0.7464
Best model saved.


Epoch 3/20: 100%|██████████| 403/403 [10:13<00:00,  1.52s/it]


...

Train Loss: 0.4320
Val Loss: 1.5969
Val Acc: 0.4836
No improvement (2/4)


Epoch 6/20: 100%|██████████| 403/403 [12:03<00:00,  1.80s/it]


Train Loss: 0.4275
Val Loss: 0.6077
Val Acc: 0.7607
No improvement (3/4)


Epoch 7/20: 100%|██████████| 403/403 [11:48<00:00,  1.76s/it]


Train Loss: 0.3867
Val Loss: 0.5160
Val Acc: 0.7874
No improvement (4/4)
Early stopping triggered.


In [9]:
torch.save(model.state_dict(),"./classification_model/classification_model.pth")

In [10]:
torch.save(model,"./classification_model/full_classification_model.pth")

In [11]:
torch.jit.script(model).save("./classification_model/scripted_classification_model.pt")

In [12]:
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss
},"./classification_model/checkpoint.pth")

In [13]:
import pickle

with open("./classification_model/model.pkl","wb") as f:
    pickle.dump(model,f)

In [14]:
import joblib
joblib.dump(model, "./classification_model/model.joblib")

['./classification_model/model.joblib']